In [9]:
%%capture
!pip install unsloth
!pip install --upgrade xformers trl peft accelerate bitsandbytes

In [10]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("\n Model and Tokenizer loaded successfully")

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

 Model and Tokenizer loaded successfully


In [11]:
from peft import LoraConfig
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha =  16,
    lora_dropout = 0,
    bias = "none",

    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("\n LoRA Adapters successfully attached!")


 LoRA Adapters successfully attached!


In [12]:
from datasets import load_dataset

finance_prompt = """You are a highly analytical Quantitative Finance AI.
Analyze the following financial news headline and determine its sentiment.
Respond ONLY with one of the following labels: Positive, Negative, or Neutral.

### News Headline:
{}

### Sentiment:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
  inputs = examples["text"]
  labels = examples["label"]

  label_map = {0: "Negative", 1: "Positive", 2: "Neutral"}

  texts = []
  for text, label_idx in zip(inputs, labels):
    sentiment_word = label_map[label_idx]

    formatted_text = finance_prompt.format(text, sentiment_word) + EOS_TOKEN
    texts.append(formatted_text)

  return { "text" : texts, }

print("Downloading Financial PhraseBank...")
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment", split="train")

formatted_dataset = dataset.map(formatting_prompts_func, batched = True,)

print(f"\n Successfully formatted {len(formatted_dataset)} financial statements!")
print("---Example Data---")
print(formatted_dataset[0]["text"])


 Successfully formatted 9543 financial statements!
---Example Data---
You are a highly analytical Quantitative Finance AI.
Analyze the following financial news headline and determine its sentiment.
Respond ONLY with one of the following labels: Positive, Negative, or Neutral.

### News Headline:
$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT

### Sentiment:
Negative<|eot_id|>


In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,

    args = TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Starting the training loop... Watch the 'Loss' column go down!")
trainer_stats = trainer.train()
print("Training Complete!")

Starting the training loop... Watch the 'Loss' column go down!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,543 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,3.530200
2,3.528500
3,3.432900
4,3.496000
5,3.116800
6,2.910700
7,2.439500
8,2.090400
9,1.736200
10,1.453200


Training Complete!


In [ ]:
from google.colab import files
import os

print("Packaging the model into a GGUF file...")

model.save_pretrained_gguf(
    "vantage_quant_model",
    tokenizer,
    quantization_method = "q4_k_m"
)

print("Packaging complete!")

file_name = "vantage_quant_model_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf"

if os.path.exists(file_name):
    print(f"\n Starting downlod of {file_name}...")
    files.downlod(file_name)
else: 
    print(f"\n Error")

Packaging the model into a GGUF file...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:56<00:00, 88.16s/it] 


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:46<00:00, 53.45s/it]


Unsloth: Merge process complete. Saved to `/content/vantage_quant_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['vantage_quant_model_gguf/Llama-3.2-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['vantage_quant_model_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: llama.cpp/llama-cli --model vantage_quant_model_gguf/Llama-3.2-3B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to vantage_quant_model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f vantage_quant_model_gguf/Modelfile
Packaging complete!

 Error
